# 🛠️ MLE Data Engineering Masterclass
Bhai, welcome to the core of the ML pipeline. 
Yahan main tujhe scratch se dikhaunga ki hum raw data ko utha kar kaise ML ke laayak banate hain (Feature Engineering). 
Interview mein jab puche ki 'Data prep kaise ki?', toh exactly yahi steps batane hain.

## Step 1: Data Fetching (The API)
Humne **Yahoo Finance API (`yfinance`)** use ki hai. Ye free hai, reliable hai, aur institutional quality OHLCV (Open, High, Low, Close, Volume) data deti hai.

**Interview tip:** Agar pooche ki live systems mein kya use hota hai, toh bolna 'Bloomberg Terminal APIs ya Refinitiv TickHistory, but for this prototype I used yfinance'.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Fetch 2 years of daily data for SPY
ticker = 'SPY'
raw_df = yf.download(ticker, period='2y', auto_adjust=True, progress=False)

# 2. Clean MultiIndex columns (yfinance sometimes returns nested columns)
if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df.columns = raw_df.columns.get_level_values(0)

# 3. Drop holidays/weekends where market is closed
df = raw_df.dropna().copy()
df.tail()

## Step 2: Log Returns (Not Absolute Prices)
ML models absolute prices (jaise $745) samajh nahi paate. Unhe sirf % change samajh aata hai. 
Hum normal percentage change ki jagah **Log Returns** use karte hain.

**Kyun?** Kyuki log returns *time-additive* hote hain. Agar aaj +10% hua aur kal -10%, normal math me tu lose karta hai. Log math me wo exactly 0 hota hai. Statistical models log returns par hi build hote hain.

In [ ]:
# Formula: ln(Current Close / Previous Close)
df['Return'] = np.log(df['Close'] / df['Close'].shift(1))

df[['Close', 'Return']].tail()

## Step 3: Defining the Target Variable (Y)
Supervised ML mein ek 'Target' chahiye hota hai jise predict karna hai. 
Hum chahte hain model predict kare ki **agle 5 din mein kitni volatility (risk) aayegi**.

**Data Leakage Warning ⚠️:** 
Hum future predict kar rahe hain, isliye `.shift(-1)` lagana zaroori hai. Iska matlab hai ki hum agle din se aage ke 5 din dekh rahe hain. Agar shift nahi kiya, toh model purana data dekh lega aur 'cheating' karega. Ise MLE mein *Data Leakage* kehte hain.

In [ ]:
# 5-day Forward Realized Volatility (Annualised)
# sqrt(252) is used because there are 252 trading days in a year.
df['Target_Vol_5d'] = df['Return'].shift(-1).rolling(5).std() * np.sqrt(252)

df[['Return', 'Target_Vol_5d']].tail(10)

## Step 4: Feature Engineering (X)
Ab hum input features banayenge jo model ko market ka 'Mood' batayenge.

In [ ]:
# Feature 1: Lagged Returns (Memory of past days)
df['ret_lag_1'] = df['Return'].shift(1)
df['ret_lag_5'] = df['Return'].shift(5)

# Feature 2: Historical Volatility (How crazy was the market in the last 20 days?)
df['rv_20d'] = df['Return'].rolling(20).std() * np.sqrt(252)

# Feature 3: RSI (Relative Strength Index) - Detects if market is tired of going up/down
delta = df['Close'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss + 1e-8)
df['rsi_14'] = 100 - (100 / (1 + rs))

# Feature 4: Bollinger Band Width - Volatility Squeeze indicator
sma20 = df['Close'].rolling(20).mean()
std20 = df['Close'].rolling(20).std()
df['bb_width'] = (2 * std20) / (sma20 + 1e-8)

df[['rv_20d', 'rsi_14', 'bb_width']].tail()

## Step 5: Final Clean Up
Jab hum `.shift(1)` ya `.rolling(20)` karte hain, toh pehle 20 din ka data `NaN` (Not a Number) ban jata hai kyuki past history available nahi hoti.
Usi tarah target mein `.shift(-1)` ki wajah se ekdum aakhri din `NaN` ban jata hai kyuki future abhi aaya nahi.

In sabhi NaNs ko drop karna zaruri hai, warna Scikit-Learn ka `GradientBoostingRegressor` error phek dega.

In [ ]:
# Final dataset ready for ML
ml_dataset = df.dropna().copy()

print(f"Raw Data Rows: {len(raw_df)}")
print(f"Clean ML Rows: {len(ml_dataset)}")

ml_dataset[['Close', 'rv_20d', 'rsi_14', 'bb_width', 'Target_Vol_5d']].tail()

## Step 6: ML Training Logic (How the Backend does it)
Ab backend is `ml_dataset` ko leta hai.
1. **X** = ['ret_lag_1', 'rv_20d', 'rsi_14', 'bb_width']
2. **Y** = 'Target_Vol_5d'
3. Hum `TimeSeriesSplit` se data ko parts mein baant-te hain (Future mein se train karna allowed nahi hai).
4. `GradientBoostingRegressor(n_estimators=100, max_depth=3)` train hota hai.
5. Model predict karta hai, aur RMSE calculate hota hai jo tu page pe dekhta hai (e.g., 6.48%).